# Active Learning Experiment: Entropy vs Random

**Dataset**: Russian Financial News (`data/labeled/strategy_a_labeled.csv`)  
**Task**: Binary classification — `positive_market_impact` (0/1)  
**Model**: LogisticRegression + TF-IDF(10k features, bigrams)  
**Strategies**: Entropy sampling vs Random sampling  
**Config**: init=200, batch=100, iterations=10

In [1]:
import sys, json, os
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
os.makedirs('../data/reports', exist_ok=True)
print('OK')


OK


## 1. Load results

In [2]:
with open('../data/reports/al_results.json') as f:
    results = json.load(f)

entropy_hist = pd.DataFrame(results['entropy'])
random_hist = pd.DataFrame(results['random'])

print('Config:')
for k, v in results['config'].items():
    print(f'  {k}: {v}')
print()
print('Entropy final:', entropy_hist.iloc[-1][['n_labeled','accuracy','f1']].to_dict())
print('Random  final:', random_hist.iloc[-1][['n_labeled','accuracy','f1']].to_dict())


Config:
  random_state: 42
  test_size: 0.2
  initial_labeled: 200
  batch_size: 100
  n_iterations: 10
  model: LogisticRegression
  vectorizer: TF-IDF(max_features=10000, ngram=(1,2))

Entropy final: {'n_labeled': 1200, 'accuracy': 0.5134, 'f1': 0.4539}
Random  final: {'n_labeled': 1200, 'accuracy': 0.5072, 'f1': 0.5028}


## 2. Learning curves

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for metric, ax, title in [('f1', axes[0], 'Macro F1'), ('accuracy', axes[1], 'Accuracy')]:
    ax.plot(entropy_hist['n_labeled'], entropy_hist[metric],
            'o-', color='#e07070', label='Entropy', linewidth=2, markersize=5)
    ax.plot(random_hist['n_labeled'], random_hist[metric],
            's--', color='#6699cc', label='Random', linewidth=2, markersize=5)
    ax.set_title(f'Learning Curve: {title}', fontsize=13)
    ax.set_xlabel('Number of labeled examples')
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('../data/reports/learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved learning_curves.png')


Saved learning_curves.png


## 3. Comparison table

In [4]:
table = pd.DataFrame({
    'N labeled':     entropy_hist['n_labeled'].values,
    'Entropy F1':    entropy_hist['f1'].values,
    'Entropy Acc':   entropy_hist['accuracy'].values,
    'Random F1':     random_hist['f1'].values,
    'Random Acc':    random_hist['accuracy'].values,
    'Delta F1':      (entropy_hist['f1'] - random_hist['f1']).values,
})
table = table.round(4)
print(table.to_string(index=False))
table


 N labeled  Entropy F1  Entropy Acc  Random F1  Random Acc  Delta F1
       200      0.4609       0.5094     0.4609      0.5094    0.0000
       300      0.4723       0.5011     0.4957      0.4996   -0.0234
       400      0.4728       0.5065     0.4974      0.5011   -0.0246
       500      0.4687       0.5047     0.4965      0.5029   -0.0278
       600      0.4762       0.5043     0.4951      0.5007   -0.0189
       700      0.4703       0.5004     0.4904      0.5004   -0.0201
       800      0.4677       0.4967     0.5027      0.5036   -0.0350
       900      0.4633       0.5051     0.5072      0.5072   -0.0439
      1000      0.4665       0.5141     0.5109      0.5109   -0.0444
      1100      0.4494       0.5112     0.5066      0.5069   -0.0572
      1200      0.4539       0.5134     0.5028      0.5072   -0.0489


,N labeled,Entropy F1,Entropy Acc,Random F1,Random Acc,Delta F1
0,200,0.4609,0.5094,0.4609,0.5094,0.0000
1,300,0.4723,0.5011,0.4957,0.4996,-0.0234
2,400,0.4728,0.5065,0.4974,0.5011,-0.0246
3,500,0.4687,0.5047,0.4965,0.5029,-0.0278
4,600,0.4762,0.5043,0.4951,0.5007,-0.0189
5,700,0.4703,0.5004,0.4904,0.5004,-0.0201
6,800,0.4677,0.4967,0.5027,0.5036,-0.0350
7,900,0.4633,0.5051,0.5072,0.5072,-0.0439
8,1000,0.4665,0.5141,0.5109,0.5109,-0.0444
9,1100,0.4494,0.5112,0.5066,0.5069,-0.0572


## 4. Confusion matrices (final iteration)

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, hist, title in [
    (axes[0], results['entropy'], 'Entropy (final)'),
    (axes[1], results['random'],  'Random (final)')
]:
    cm = np.array(hist[-1]['confusion_matrix'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Neg/Neutral', 'Positive'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{title}\nF1={hist[-1]["f1"]:.4f}', fontsize=12)

plt.tight_layout()
plt.savefig('../data/reports/confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved confusion_matrices.png')


Saved confusion_matrices.png


## 5. Key finding: why Entropy < Random

In [6]:
print('=== KEY FINDING ===')
print()
print('Random sampling OUTPERFORMS Entropy sampling on this dataset.')
print()
print('Likely causes:')
print('1. Label noise (~20% from GPT-4o silver labels)')
print('   Entropy sampling selects the most uncertain examples,')
print('   which in a noisy dataset are often the NOISIEST, not the most informative.')
print()
print('2. Short texts (many Telegram posts < 100 chars)')
print('   TF-IDF has limited signal; uncertainty estimates are unreliable.')
print()
print('3. Overall low F1 (~0.50) indicates the features are not discriminative enough')
print('   for entropy to provide advantage over random selection.')
print()
print(f'Entropy final F1:  {results["entropy"][-1]["f1"]:.4f}')
print(f'Random  final F1:  {results["random"][-1]["f1"]:.4f}')
print(f'Delta:             {results["entropy"][-1]["f1"] - results["random"][-1]["f1"]:+.4f}')


=== KEY FINDING ===

Random sampling OUTPERFORMS Entropy sampling on this dataset.

Likely causes:
1. Label noise (~20% from GPT-4o silver labels)
   Entropy sampling selects the most uncertain examples,
   which in a noisy dataset are often the NOISIEST, not the most informative.

2. Short texts (many Telegram posts < 100 chars)
   TF-IDF has limited signal; uncertainty estimates are unreliable.

3. Overall low F1 (~0.50) indicates the features are not discriminative enough
   for entropy to provide advantage over random selection.

Entropy final F1:  0.4539
Random  final F1:  0.5028
Delta:             -0.0489


## 6. Score distribution by strategy

In [7]:
fig, ax = plt.subplots(figsize=(12, 4))
x = entropy_hist['n_labeled'].values
ax.fill_between(x, entropy_hist['f1'], alpha=0.3, color='#e07070', label='Entropy F1')
ax.fill_between(x, random_hist['f1'],  alpha=0.3, color='#6699cc', label='Random F1')
ax.plot(x, entropy_hist['f1'], 'o-', color='#e07070', linewidth=2)
ax.plot(x, random_hist['f1'],  's-', color='#6699cc', linewidth=2)
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.7, label='F1=0.5 (baseline)')
ax.set_title('F1 Score: Entropy vs Random across all iterations', fontsize=13)
ax.set_xlabel('Number of labeled examples')
ax.set_ylabel('Macro F1')
ax.legend()
plt.tight_layout()
plt.savefig('../data/reports/score_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved score_distributions.png')


Saved score_distributions.png


## 7. Summary

In [8]:
e = results['entropy'][-1]
r = results['random'][-1]
print('='*50)
print('EXPERIMENT SUMMARY')
print('='*50)
print(f'Iterations:     {results["config"]["n_iterations"]}')
print(f'Final N labeled: {e["n_labeled"]}')
print()
print(f'Entropy  F1={e["f1"]:.4f}  Acc={e["accuracy"]:.4f}')
print(f'Random   F1={r["f1"]:.4f}  Acc={r["accuracy"]:.4f}')
print(f'Winner: Random (Delta F1 = {r["f1"]-e["f1"]:+.4f})')
print()
print('Recommendation: with noisy silver labels, random sampling is more robust.')
print('To benefit from entropy sampling: clean labels or use a stronger feature extractor.')


EXPERIMENT SUMMARY
Iterations:     10
Final N labeled: 1200

Entropy  F1=0.4539  Acc=0.5134
Random   F1=0.5028  Acc=0.5072
Winner: Random (Delta F1 = +0.0489)

Recommendation: with noisy silver labels, random sampling is more robust.
To benefit from entropy sampling: clean labels or use a stronger feature extractor.
